# Tabular — โจทย์แบบ "ทำนายตัวเลขจากตาราง" (Regression)

ข้อมูลตาราง -> ทำนายตัวเลขต่อเนื่อง (เช่น ราคาบ้าน, คะแนนความเสี่ยง, ยอดขาย)

วิธีในโน้ตบุ๊กนี้:
- วิธีที่ 1 (ง่ายสุด) = `AutoGluon` แค่บอก problem_type='regression'
- วิธีที่ 2 (ไม่บังคับ) = `LightGBM` regressor + cross-validation


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อคำตอบเป็น "ตัวเลขต่อเนื่อง" ไม่ใช่หมวด metric มักเป็น `RMSE` หรือ `MAE`
ถ้าคำตอบเป็น "หมวด/ป้าย" -> ไปใช้ `classification.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, ไฟล์ csv, `TARGET`, `METRIC`

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U "autogluon.tabular[all]"
!pip -q install lightgbm scikit-learn pandas numpy

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "ใส่-slug-ของ-competition-regression-ตรงนี้"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — โหลดข้อมูล + CONFIG

In [ ]:
import os, pandas as pd, numpy as np
TRAIN_CSV=os.path.join(DATA_DIR,"train.csv"); TEST_CSV=os.path.join(DATA_DIR,"test.csv")
SAMPLE_SUB=os.path.join(DATA_DIR,"sample_submission.csv")
train=pd.read_csv(TRAIN_CSV); test=pd.read_csv(TEST_CSV); sample=pd.read_csv(SAMPLE_SUB)
ID_COL=sample.columns[0]   # เดาอัตโนมัติ: คอลัมน์แรกของ sample คือ id
TARGET=sample.columns[1]   # << แก้ถ้าผิด: คอลัมน์ตัวเลขที่ทำนาย เช่น "SalePrice", "price", "target"
# << แก้ METRIC ตาม tab Evaluation: Kaggle เขียน RMSE ใส่ "root_mean_squared_error", MAE ใส่ "mean_absolute_error", R2 ใส่ "r2"
METRIC="root_mean_squared_error"
print("คอลัมน์:",list(train.columns)); display(train.head()); display(sample.head())

## 🔎 โจทย์นี้ต้องส่งอะไร + วัดด้วยอะไร (รันก่อนทำ submission)

เซลล์นี้ตอบ 3 คำถามที่มือใหม่มักไม่รู้:
- ต้องเติม "คอลัมน์อะไร" ลง submission และเป็น "ชนิดไหน" (ดูจาก sample_submission)
- โจทย์วัดด้วย "metric อะไร" (ดึงจาก Kaggle ให้อัตโนมัติ)
- ต้องส่งเป็น "ป้าย / ความน่าจะเป็น / ตัวเลข" แบบไหน

In [ ]:
# บอกว่าต้องเติมอะไรลง submission + ตัวอย่างค่าที่ sample ให้มา
print("ไฟล์ส่งต้องมีคอลัมน์:", list(sample.columns), "| จำนวนแถว:", len(sample))
for _c in list(sample.columns)[1:]:
    print(f"  - เติม '{_c}': ชนิด {sample[_c].dtype}, ตัวอย่างค่าใน sample = {list(sample[_c].dropna().unique()[:3])}")
# ดึง metric จาก Kaggle อัตโนมัติ (ถ้าต่อ API ได้)
_metric = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    _api = KaggleApi(); _api.authenticate()
    _resp = _api.competitions_list(search=COMP)
    for _co in (getattr(_resp, "competitions", _resp) or []):   # kaggle ใหม่คืน response (ใช้ .competitions), เก่าคืน list
        if str(getattr(_co, "ref", "")).rstrip("/").split("/")[-1] == COMP:
            _metric = getattr(_co, "evaluation_metric", None) or getattr(_co, "evaluationMetric", None); break
except Exception:
    pass
def _how_to_send(m):
    m = (m or "").lower()
    if any(k in m for k in ["auc","roc","log loss","logloss","brier","probab"]): return "ความน่าจะเป็น (เลข 0-1)"
    if any(k in m for k in ["rmse","mae","mse","r2","rmsle","error","mape","smape"]): return "ตัวเลขต่อเนื่อง"
    return "ป้าย/คลาส (เช่น 0/1 หรือชื่อคลาส)"
print("\nMetric (ดึงจาก Kaggle):", _metric or "ดึงไม่ได้ -> เปิด tab Evaluation บนหน้า Kaggle อ่านเอง")
print("=> ปกติต้องส่งเป็น:", _how_to_send(_metric), "(เช็คกับ tab Evaluation อีกที)")

## วิธีที่ 1 — AutoGluon Regression (ง่ายสุด)

In [ ]:
from autogluon.tabular import TabularPredictor
# Kaggle มักเขียนสั้น (RMSE/MAE/R2) แต่ AutoGluon ใช้ชื่อยาว -> แปลงให้ (ถ้าไม่รู้จัก ส่งตรง ๆ)
AG_METRIC = {"rmse":"root_mean_squared_error","mae":"mean_absolute_error","r2":"r2"}.get(METRIC.lower(), METRIC)
train_ag=train.drop(columns=[c for c in [ID_COL] if c in train.columns])
test_ag =test.drop(columns=[c for c in [ID_COL] if c in test.columns])
predictor=TabularPredictor(label=TARGET, problem_type="regression", eval_metric=AG_METRIC, path="ag_reg").fit(
    train_ag, presets="best_quality", time_limit=600)   # << แก้ time_limit: วินาที (600=10นาที) ลอง 120 ก่อน
out=sample.copy(); out[TARGET]=predictor.predict(test_ag).values
out.to_csv("submission.csv",index=False); print("บันทึก submission.csv"); display(out.head())

## วิธีที่ 2 — LightGBM Regressor + CV (ไม่บังคับ)

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
y=train[TARGET].values
X=train.drop(columns=[c for c in [ID_COL,TARGET] if c in train.columns])
Xte=test.drop(columns=[c for c in [ID_COL] if c in test.columns])
for df in (X,Xte):
    for col in df.columns:
        if df[col].dtype=="object": df[col]=df[col].astype("category")
oof=np.zeros(len(y)); pred=np.zeros(len(Xte))
for tr,va in KFold(5,shuffle=True,random_state=42).split(X):
    m=lgb.LGBMRegressor(n_estimators=2000,learning_rate=0.02,num_leaves=31,random_state=42,verbose=-1)
    m.fit(X.iloc[tr],y[tr],eval_set=[(X.iloc[va],y[va])],callbacks=[lgb.early_stopping(80,verbose=False)])
    oof[va]=m.predict(X.iloc[va]); pred+=m.predict(Xte)/5
print("OOF RMSE =", round(np.sqrt(mean_squared_error(y,oof)),4))   # sklearn 1.6 เอา squared=False ออกแล้ว ใช้ np.sqrt แทน
out=sample.copy(); out[TARGET]=pred; out.to_csv("submission_lgbm.csv",index=False)
print("บันทึก submission_lgbm.csv")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "autogluon regression"
!kaggle competitions submissions -c {COMP} | head